# 12 — Aim 2 ALT Signature: Sex-Adjusted Sensitivity Analysis

Sensitivity analysis addressing sex confounding observed in the notebook 11 primary top-20 (USP9Y and DDX3Y). Three designs × two definitions (primary / pragmatic) = 6 analyses. All analyses reuse helpers from `src.aim2_signature`; no logic is duplicated.

**Designs**
- **D1 – Sex covariate**: Append donor sex (Female=0, Male=1) as a numeric feature after the gene variance filter, before the StandardScaler. Aborts per-definition if any included sample lacks a Female/Male sex annotation.
- **D2 – No-Y / No-X-CT**: Exclude all 41 Y-chromosome gene columns plus XAGE1A, XAGE1B, MAGEA10, MAGEB2 from the feature universe before the variance filter. Standard `run_loocv` pipeline otherwise unchanged.
- **D3 – Curated ALT-only**: Restrict to the 21-gene curated ALT gene list (19/21 found in matrix). No variance filter; StandardScaler → ElasticNet only.

In [1]:
import sys, random, time, warnings
from pathlib import Path

import numpy as np

random.seed(42)
np.random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

from src.utils import log_versions, sha256_file
import pandas as pd

log_versions(['pandas', 'numpy', 'scipy', 'sklearn', 'joblib'])
print(f'Run: {RUN_TIMESTAMP}')


ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  sklearn: 1.8.0
  joblib: 1.5.3
Run: 2026-06-05T17:10:00Z


## 1 — Imports and configuration

In [2]:
from src.aim2_signature import (
    apply_inclusion_filter, assert_expected_counts, assert_ground_truth,
    direction_counts, labels_from_direction, load_expression,
    run_loocv, evaluate, bootstrap_ci, permutation_test, refit_full,
    ranked_genes, top_genes, flag_alt_genes,
    EXPECTED_COUNTS, ALT_GENE_LIST, C_GRID,
    TOP_K_VARIABLE_GENES, N_PERMUTATIONS, N_BOOTSTRAP, RANDOM_SEED, L1_RATIO,
    INNER_CV_FOLDS, strip_gene_symbol,
    VarianceGeneFilter, build_estimator, LoocvResult,
)
from src.aim1_inventory import DEPMAP_RNA_PATH, DEPMAP_RNA_DATA_TYPE, depmap_rna_model_ids
from sklearn.model_selection import GridSearchCV, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.preprocessing import StandardScaler
import joblib, json

MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
depmap_models = depmap_rna_model_ids(DEPMAP_RNA_PATH)
DEFINITIONS = ['primary', 'pragmatic']

print(f'Manifest rows: {len(manifest)}')
print(f'DepMap RNA ModelIDs: {len(depmap_models)}')
print(f'RNA matrix: {DEPMAP_RNA_PATH.relative_to(ROOT)}  [{DEPMAP_RNA_DATA_TYPE}]')
print(f'Config: top_k={TOP_K_VARIABLE_GENES}, n_perm={N_PERMUTATIONS}, n_boot={N_BOOTSTRAP}, seed={RANDOM_SEED}')


Manifest rows: 115
DepMap RNA ModelIDs: 1719
RNA matrix: data/raw/depmap/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv  [log2(TPM+1) protein-coding]
Config: top_k=500, n_perm=100, n_boot=1000, seed=42


## 2 — Inclusion filters and sex annotation

In [3]:
filtered, traces = {}, {}
for defn in DEFINITIONS:
    df, trace = apply_inclusion_filter(manifest, depmap_models, defn)
    assert_expected_counts(df, defn)
    assert_ground_truth(df)
    filtered[defn], traces[defn] = df, trace
    print(f'[{defn}] N={len(df)}  counts={direction_counts(df)}')

print('Inclusion filter + ground-truth assertions PASSED.')


[primary] N=20  counts={'positive': 12, 'negative': 8}
[pragmatic] N=31  counts={'positive': 18, 'negative': 13}
Inclusion filter + ground-truth assertions PASSED.


In [4]:
model_csv = ROOT / 'data' / 'raw' / 'depmap' / 'Model.csv'
model_info = pd.read_csv(model_csv, dtype=str)[['ModelID', 'Sex']]
sex_lookup = dict(zip(model_info['ModelID'], model_info['Sex']))

sex_status = {}  # defn -> {ok, sex_vec, unknown, sexes}
for defn in DEFINITIONS:
    ids = filtered[defn]['sample_id'].tolist()
    sexes = [sex_lookup.get(sid, float('nan')) for sid in ids]
    unknown_sex = [ids[i] for i, s in enumerate(sexes) if s not in ('Female', 'Male')]
    sex_numeric = np.array(
        [1.0 if s == 'Male' else (0.0 if s == 'Female' else np.nan) for s in sexes]
    )
    sex_status[defn] = {
        'ok': len(unknown_sex) == 0,
        'sex_vec': sex_numeric,
        'unknown': unknown_sex,
        'sexes': sexes,
    }
    if unknown_sex:
        print(f'[{defn}] WARNING: sex covariate will be SKIPPED — '
              f'{len(unknown_sex)} sample(s) lack Female/Male annotation: {unknown_sex}')
    else:
        n_f = sum(s == 'Female' for s in sexes)
        n_m = sum(s == 'Male' for s in sexes)
        print(f'[{defn}] Sex annotation complete — {n_f} Female / {n_m} Male')


[primary] Sex annotation complete — 11 Female / 9 Male
[pragmatic] WARNING: sex covariate will be SKIPPED — 4 sample(s) lack Female/Male annotation: ['ACH-001813', 'ACH-001817', 'ACH-002779', 'ACH-003573']


## 3 — Load expression matrix

In [5]:
print('Loading expression (pragmatic superset, then align per definition) ...')
t0 = time.time()
expr_all = load_expression(filtered['pragmatic']['sample_id'].tolist())
gene_cols = list(expr_all.columns)
print(f'Expression matrix: {expr_all.shape[0]} samples x {expr_all.shape[1]} genes  '
      f'({time.time()-t0:.0f}s)')

X_all, y_all, ids_all = {}, {}, {}
for defn in DEFINITIONS:
    ids = filtered[defn]['sample_id'].tolist()
    X_all[defn] = expr_all.loc[ids].to_numpy(dtype=float)
    y_all[defn] = labels_from_direction(filtered[defn])
    ids_all[defn] = ids
    print(f'[{defn}] X={X_all[defn].shape}  '
          f'pos={y_all[defn].sum()}  neg={(y_all[defn]==0).sum()}')


Loading expression (pragmatic superset, then align per definition) ...
Expression matrix: 31 samples x 19215 genes  (8s)
[primary] X=(20, 19215)  pos=12  neg=8
[pragmatic] X=(31, 19215)  pos=18  neg=13


## 4 — Custom LOOCV helpers

Sex-covariate and curated-ALT designs require non-standard pipelines. These helpers reuse `VarianceGeneFilter`, `build_estimator`, `LoocvResult`, and constants imported from `src.aim2_signature`. No logic is duplicated.

In [6]:
def _inner_cv_splits(n_pos, n_neg, random_state=RANDOM_SEED):
    n_splits = max(2, min(INNER_CV_FOLDS, n_pos, n_neg))
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)


def run_loocv_sex_covariate(X_expr, sex_vec, y, sample_ids,
                             top_k=TOP_K_VARIABLE_GENES,
                             random_state=RANDOM_SEED, fixed_C=None):
    """Outer LOOCV: gene VarianceGeneFilter -> [append sex] -> StandardScaler -> LR.

    Sex bypasses the gene variance filter and is appended after fold-local gene
    selection. Inner C selection uses a StandardScaler+LR Pipeline so scaling
    stays fold-local within each inner fold.
    """
    X_expr = np.asarray(X_expr, dtype=float)
    sex_vec = np.asarray(sex_vec, dtype=float)
    y = np.asarray(y, dtype=int)
    loo = LeaveOneOut()
    probs = np.empty(len(y), dtype=float)
    selected_Cs = []

    for train_idx, test_idx in loo.split(X_expr):
        # Fold-local gene variance filter
        vf = VarianceGeneFilter(top_k=top_k)
        vf.fit(X_expr[train_idx])
        Xg_tr = vf.transform(X_expr[train_idx])
        Xg_te = vf.transform(X_expr[test_idx])
        # Append sex covariate (not filtered)
        X_tr_raw = np.hstack([Xg_tr, sex_vec[train_idx].reshape(-1, 1)])
        X_te_raw = np.hstack([Xg_te, sex_vec[test_idx].reshape(-1, 1)])
        # Inner C selection — pipeline re-scales within each inner fold
        if fixed_C is not None:
            best_C = float(fixed_C)
        else:
            n_pos_tr = int(y[train_idx].sum())
            n_neg_tr = int(len(train_idx) - n_pos_tr)
            inner_pipe = SKPipeline([
                ('scaler', StandardScaler()),
                ('clf', build_estimator(C=1.0, random_state=random_state)),
            ])
            cv = _inner_cv_splits(n_pos_tr, n_neg_tr, random_state)
            grid = GridSearchCV(inner_pipe, {'clf__C': list(C_GRID)},
                                scoring='roc_auc', cv=cv, n_jobs=1, refit=False)
            grid.fit(X_tr_raw, y[train_idx])
            best_C = float(grid.best_params_['clf__C'])
        # Outer refit with fold-local scaler
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr_raw)
        X_te = sc.transform(X_te_raw)
        clf = build_estimator(C=best_C, random_state=random_state)
        clf.fit(X_tr, y[train_idx])
        probs[test_idx[0]] = clf.predict_proba(X_te)[:, 1][0]
        selected_Cs.append(best_C)

    return LoocvResult(list(sample_ids), y, probs, selected_Cs, [])


def run_loocv_no_filter(X, y, sample_ids, random_state=RANDOM_SEED, fixed_C=None):
    """Outer LOOCV with StandardScaler only (no gene variance filter).

    For curated gene sets where all features are meaningful regardless of variance.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    loo = LeaveOneOut()
    probs = np.empty(len(y), dtype=float)
    selected_Cs = []

    for train_idx, test_idx in loo.split(X):
        X_tr_raw, X_te_raw = X[train_idx], X[test_idx]
        if fixed_C is not None:
            best_C = float(fixed_C)
        else:
            n_pos_tr = int(y[train_idx].sum())
            n_neg_tr = int(len(train_idx) - n_pos_tr)
            inner_pipe = SKPipeline([
                ('scaler', StandardScaler()),
                ('clf', build_estimator(C=1.0, random_state=random_state)),
            ])
            cv = _inner_cv_splits(n_pos_tr, n_neg_tr, random_state)
            grid = GridSearchCV(inner_pipe, {'clf__C': list(C_GRID)},
                                scoring='roc_auc', cv=cv, n_jobs=1, refit=False)
            grid.fit(X_tr_raw, y[train_idx])
            best_C = float(grid.best_params_['clf__C'])
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr_raw)
        X_te = sc.transform(X_te_raw)
        clf = build_estimator(C=best_C, random_state=random_state)
        clf.fit(X_tr, y[train_idx])
        probs[test_idx[0]] = clf.predict_proba(X_te)[:, 1][0]
        selected_Cs.append(best_C)

    return LoocvResult(list(sample_ids), y, probs, selected_Cs, [])


def permutation_test_fn(run_fn_with_y, y, observed_auroc, fixed_C,
                         n_perm=N_PERMUTATIONS, seed=RANDOM_SEED, progress_every=20):
    """Label-shuffling permutation null for custom LOOCV functions.

    run_fn_with_y(y, fixed_C) must return a LoocvResult.
    Fixed C is the median LOOCV-selected C from the observed run.
    """
    rng = np.random.default_rng(seed)
    perm_aurocs = []
    t0 = time.time()
    for i in range(n_perm):
        y_perm = rng.permutation(y)
        result = run_fn_with_y(y_perm, fixed_C)
        perm_aurocs.append(result.auroc)
        if progress_every and ((i + 1) % progress_every == 0 or i + 1 == n_perm):
            print(f'    permutation {i+1}/{n_perm}  ({time.time()-t0:.0f}s)', flush=True)
    perm_arr = np.asarray(perm_aurocs)
    return {
        'observed_auroc': float(observed_auroc),
        'perm_aurocs': perm_arr,
        'p_value': float(np.mean(perm_arr >= observed_auroc)),
        'n_perm': n_perm,
        'permutation_fixed_C': float(fixed_C),
        'select_C_per_fold': False,
    }


print('Helpers defined: run_loocv_sex_covariate, run_loocv_no_filter, permutation_test_fn')


Helpers defined: run_loocv_sex_covariate, run_loocv_no_filter, permutation_test_fn


## 5 — Design 1: Sex covariate

Donor sex appended as a numeric feature (Female=0, Male=1) after the fold-local gene variance filter, before the StandardScaler. Inner C-selection and outer scaler are both fold-local. Aborts per-definition if any included sample has sex other than Female or Male (Unknown counts as missing).

In [7]:
sex_cov_loocv, sex_cov_metrics, sex_cov_cis, sex_cov_perms = {}, {}, {}, {}

for defn in DEFINITIONS:
    print(f'\n=== D1 Sex covariate [{defn.upper()}] ===')
    if not sex_status[defn]['ok']:
        unk = sex_status[defn]['unknown']
        print(f'  SKIPPED — {len(unk)} sample(s) lack Female/Male annotation: {unk}')
        sex_cov_loocv[defn] = None
        sex_cov_metrics[defn] = None
        sex_cov_cis[defn] = None
        sex_cov_perms[defn] = None
        continue

    X_expr = X_all[defn]
    sex_vec = sex_status[defn]['sex_vec']
    y = y_all[defn]
    ids = ids_all[defn]

    print(f'  LOOCV (N={len(y)}, pos={y.sum()}, neg={(y==0).sum()}) ...')
    t0 = time.time()
    res = run_loocv_sex_covariate(X_expr, sex_vec, y, ids)
    print(f'  done in {time.time()-t0:.0f}s  AUROC={res.auroc:.4f}  '
          f'median_C={float(np.median(res.selected_Cs)):.4f}')

    metrics = evaluate(res.y_true, res.y_prob)
    cis = bootstrap_ci(res.y_true, res.y_prob, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)

    fixed_C = float(np.median(res.selected_Cs))
    print(f'  Permutation test (n={N_PERMUTATIONS}, fixed_C={fixed_C:.4f}) ...')
    _X_e, _sv, _ids = X_expr, sex_vec, ids  # explicit capture for closure
    perms = permutation_test_fn(
        run_fn_with_y=lambda y_, C_, Xe=_X_e, sv=_sv, ii=_ids: (
            run_loocv_sex_covariate(Xe, sv, y_, ii, fixed_C=C_)
        ),
        y=y, observed_auroc=res.auroc, fixed_C=fixed_C,
        n_perm=N_PERMUTATIONS, seed=RANDOM_SEED,
    )

    sex_cov_loocv[defn] = res
    sex_cov_metrics[defn] = metrics
    sex_cov_cis[defn] = cis
    sex_cov_perms[defn] = perms

    ci_auroc = cis['auroc']
    print(f'  AUROC={metrics["auroc"]:.4f} [{ci_auroc[0]:.4f},{ci_auroc[1]:.4f}]  '
          f'AUPRC={metrics["auprc"]:.4f}  '
          f'sens@90spec={metrics["sensitivity_at_90_spec"]:.4f}  '
          f'perm_p={perms["p_value"]:.3f}')



=== D1 Sex covariate [PRIMARY] ===
  LOOCV (N=20, pos=12, neg=8) ...
  done in 52s  AUROC=0.5000  median_C=31.6228
  Permutation test (n=100, fixed_C=31.6228) ...
    permutation 20/100  (64s)
    permutation 40/100  (126s)
    permutation 60/100  (191s)
    permutation 80/100  (257s)
    permutation 100/100  (322s)
  AUROC=0.5000 [0.1904,0.8001]  AUPRC=0.5818  sens@90spec=0.0000  perm_p=0.530

=== D1 Sex covariate [PRAGMATIC] ===
  SKIPPED — 4 sample(s) lack Female/Male annotation: ['ACH-001813', 'ACH-001817', 'ACH-002779', 'ACH-003573']


## 6 — Design 2: Exclude Y-chromosome + X CT antigen genes

Remove 41 Y-chromosome gene columns and 4 X-linked cancer/testis antigen columns (XAGE1A, XAGE1B, MAGEA10, MAGEB2) from the feature universe **before** the variance filter. The standard `run_loocv` pipeline is otherwise unchanged.

In [8]:
# 41 Y-chromosome gene symbols found in the DepMap RNA matrix
Y_CHR_SYMBOLS = frozenset([
    'AMELY', 'BPY2', 'BPY2B', 'BPY2C',
    'CDY1', 'CDY1B', 'CDY2A', 'CDY2B',
    'DAZ1', 'DAZ2', 'DAZ3', 'DAZ4',
    'DDX3Y', 'EIF1AY', 'HSFY1', 'HSFY2',
    'KDM5D', 'NLGN4Y', 'PCDH11Y', 'PRY', 'PRY2',
    'RBMY1A1', 'RBMY1B', 'RBMY1D', 'RBMY1E', 'RBMY1F', 'RBMY1J',
    'RPS4Y1', 'RPS4Y2', 'SRY', 'TBL1Y', 'TMSB4Y',
    'TSPY1', 'TSPY2', 'TSPY3', 'TSPY4',
    'USP9Y', 'UTY', 'VCY', 'VCY1B', 'ZFY',
])

# X-linked cancer/testis antigens that appeared in nb11 confounders
X_CT_SYMBOLS = frozenset(['MAGEB2', 'MAGEA10', 'XAGE1A', 'XAGE1B'])

EXCLUDE_SYMBOLS = Y_CHR_SYMBOLS | X_CT_SYMBOLS

gene_sym_all = [strip_gene_symbol(c) for c in gene_cols]
exclude_idx = [i for i, sym in enumerate(gene_sym_all) if sym in EXCLUDE_SYMBOLS]
exclude_names = [gene_cols[i] for i in exclude_idx]
keep_idx = [i for i in range(len(gene_cols)) if i not in set(exclude_idx)]
gene_cols_noy = [gene_cols[i] for i in keep_idx]

n_ychr_found = sum(strip_gene_symbol(c) in Y_CHR_SYMBOLS for c in gene_cols)
n_xct_found  = sum(strip_gene_symbol(c) in X_CT_SYMBOLS for c in gene_cols)
print(f'Y-chr found: {n_ychr_found} / {len(Y_CHR_SYMBOLS)}')
print(f'X CT found:  {n_xct_found} / {len(X_CT_SYMBOLS)}')
print(f'Total excluded: {len(exclude_idx)} columns')
print(f'Gene pool after exclusion: {len(gene_cols_noy)} / {len(gene_cols)}')

noy_loocv, noy_metrics, noy_cis, noy_perms = {}, {}, {}, {}

for defn in DEFINITIONS:
    print(f'\n=== D2 No-Y/no-X-CT [{defn.upper()}] ===')
    X_noy = X_all[defn][:, keep_idx]
    y = y_all[defn]
    ids = ids_all[defn]

    print(f'  LOOCV (N={len(y)}, pool={X_noy.shape[1]}) ...')
    t0 = time.time()
    res = run_loocv(X_noy, y, ids, top_k=TOP_K_VARIABLE_GENES, random_state=RANDOM_SEED)
    print(f'  done in {time.time()-t0:.0f}s  AUROC={res.auroc:.4f}  '
          f'median_C={float(np.median(res.selected_Cs)):.4f}')

    metrics = evaluate(res.y_true, res.y_prob)
    cis = bootstrap_ci(res.y_true, res.y_prob, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)

    fixed_C = float(np.median(res.selected_Cs))
    print(f'  Permutation test (n={N_PERMUTATIONS}, fixed_C={fixed_C:.4f}) ...')
    perms = permutation_test(
        X_noy, y, ids, observed_auroc=res.auroc,
        n_perm=N_PERMUTATIONS, top_k=TOP_K_VARIABLE_GENES,
        random_state=RANDOM_SEED, seed=RANDOM_SEED,
        fixed_C=fixed_C, select_C_per_fold=False,
        progress_every=20,
    )

    noy_loocv[defn] = res
    noy_metrics[defn] = metrics
    noy_cis[defn] = cis
    noy_perms[defn] = perms

    ci_auroc = cis['auroc']
    print(f'  AUROC={metrics["auroc"]:.4f} [{ci_auroc[0]:.4f},{ci_auroc[1]:.4f}]  '
          f'AUPRC={metrics["auprc"]:.4f}  '
          f'sens@90spec={metrics["sensitivity_at_90_spec"]:.4f}  '
          f'perm_p={perms["p_value"]:.3f}')


Y-chr found: 41 / 41
X CT found:  4 / 4
Total excluded: 45 columns
Gene pool after exclusion: 19170 / 19215

=== D2 No-Y/no-X-CT [PRIMARY] ===
  LOOCV (N=20, pool=19170) ...
  done in 54s  AUROC=0.3958  median_C=31.6228
  Permutation test (n=100, fixed_C=31.6228) ...
    permutation 20/100  (63s elapsed)
    permutation 40/100  (124s elapsed)
    permutation 60/100  (188s elapsed)
    permutation 80/100  (252s elapsed)
    permutation 100/100  (3357s elapsed)
  AUROC=0.3958 [0.1098,0.7144]  AUPRC=0.5386  sens@90spec=0.0000  perm_p=0.660

=== D2 No-Y/no-X-CT [PRAGMATIC] ===
  LOOCV (N=31, pool=19170) ...
  done in 271s  AUROC=0.7051  median_C=31.6228
  Permutation test (n=100, fixed_C=31.6228) ...
    permutation 20/100  (133s elapsed)
    permutation 40/100  (264s elapsed)
    permutation 60/100  (403s elapsed)
    permutation 80/100  (537s elapsed)
    permutation 100/100  (678s elapsed)
  AUROC=0.7051 [0.4772,0.8955]  AUPRC=0.7443  sens@90spec=0.2222  perm_p=0.060


## 7 — Design 3: Curated ALT-only

Restrict the feature set to the 21-gene curated ALT gene list from `src.aim2_signature.ALT_GENE_LIST`. TERC and MRE11A are absent from the DepMap RNA matrix (the column uses MRE11 instead); 19/21 genes are used. No variance filter — all curated genes pass directly to StandardScaler → ElasticNet.

In [9]:
# Identify curated ALT gene columns in the expression matrix
curated_syms_defined = list(ALT_GENE_LIST)
gene_sym_to_col = {strip_gene_symbol(c): c for c in gene_cols}

found_curated = [(sym, gene_sym_to_col[sym])
                 for sym in curated_syms_defined if sym in gene_sym_to_col]
not_found_curated = [sym for sym in curated_syms_defined if sym not in gene_sym_to_col]
curated_cols = [col for _, col in found_curated]
curated_syms = [sym for sym, _ in found_curated]
curated_col_idx = [gene_cols.index(col) for col in curated_cols]

print(f'ALT_GENE_LIST: {len(curated_syms_defined)} genes defined')
print(f'  Found in matrix ({len(found_curated)}): {curated_syms}')
print(f'  Not found ({len(not_found_curated)}): {not_found_curated}')

curated_loocv, curated_metrics, curated_cis, curated_perms = {}, {}, {}, {}
curated_models = {}

for defn in DEFINITIONS:
    print(f'\n=== D3 Curated ALT-only [{defn.upper()}] ===')
    X_cur = X_all[defn][:, curated_col_idx]
    y = y_all[defn]
    ids = ids_all[defn]

    print(f'  LOOCV (N={len(y)}, features={X_cur.shape[1]}, no variance filter) ...')
    t0 = time.time()
    res = run_loocv_no_filter(X_cur, y, ids)
    print(f'  done in {time.time()-t0:.0f}s  AUROC={res.auroc:.4f}  '
          f'median_C={float(np.median(res.selected_Cs)):.4f}')

    metrics = evaluate(res.y_true, res.y_prob)
    cis = bootstrap_ci(res.y_true, res.y_prob, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)

    fixed_C = float(np.median(res.selected_Cs))
    print(f'  Permutation test (n={N_PERMUTATIONS}, fixed_C={fixed_C:.4f}) ...')
    _X_c, _ids_c = X_cur, ids  # explicit capture for closure
    perms = permutation_test_fn(
        run_fn_with_y=lambda y_, C_, Xc=_X_c, ii=_ids_c: (
            run_loocv_no_filter(Xc, y_, ii, fixed_C=C_)
        ),
        y=y, observed_auroc=res.auroc, fixed_C=fixed_C,
        n_perm=N_PERMUTATIONS, seed=RANDOM_SEED,
    )

    # Full refit on all data for coefficient inspection
    sc_full = StandardScaler()
    X_cur_scaled = sc_full.fit_transform(X_cur)
    clf_full = build_estimator(C=fixed_C, random_state=RANDOM_SEED)
    clf_full.fit(X_cur_scaled, y)
    curated_models[defn] = {'scaler': sc_full, 'clf': clf_full}

    curated_loocv[defn] = res
    curated_metrics[defn] = metrics
    curated_cis[defn] = cis
    curated_perms[defn] = perms

    ci_auroc = cis['auroc']
    print(f'  AUROC={metrics["auroc"]:.4f} [{ci_auroc[0]:.4f},{ci_auroc[1]:.4f}]  '
          f'AUPRC={metrics["auprc"]:.4f}  '
          f'sens@90spec={metrics["sensitivity_at_90_spec"]:.4f}  '
          f'perm_p={perms["p_value"]:.3f}')


ALT_GENE_LIST: 21 genes defined
  Found in matrix (19): ['ATRX', 'DAXX', 'SMARCAL1', 'TP53', 'TERT', 'WRN', 'BLM', 'RAD50', 'NBN', 'FANCD2', 'RAD51AP1', 'WRAP53', 'RAD52', 'BRCA1', 'BRCA2', 'FANCM', 'SLX4', 'GEN1', 'MUS81']
  Not found (2): ['TERC', 'MRE11A']

=== D3 Curated ALT-only [PRIMARY] ===
  LOOCV (N=20, features=19, no variance filter) ...
  done in 2s  AUROC=0.9479  median_C=0.3162
  Permutation test (n=100, fixed_C=0.3162) ...
    permutation 20/100  (0s)
    permutation 40/100  (1s)
    permutation 60/100  (1s)
    permutation 80/100  (1s)
    permutation 100/100  (1s)
  AUROC=0.9479 [0.8125,1.0000]  AUPRC=0.9755  sens@90spec=0.9167  perm_p=0.000

=== D3 Curated ALT-only [PRAGMATIC] ===
  LOOCV (N=31, features=19, no variance filter) ...
  done in 3s  AUROC=0.8291  median_C=0.3162
  Permutation test (n=100, fixed_C=0.3162) ...
    permutation 20/100  (0s)
    permutation 40/100  (1s)
    permutation 60/100  (1s)
    permutation 80/100  (2s)
    permutation 100/100  (2s)
  A

## 8 — Notebook 11 baselines

In [10]:
nb11 = {}
for defn in DEFINITIONS:
    pred_path = PROC / f'aim2_signature_{defn}_predictions.csv'
    if not pred_path.exists():
        print(f'[WARNING] nb11 baseline not found: {pred_path.name}')
        nb11[defn] = None
        continue
    pred_df = pd.read_csv(pred_path)
    y_true = pred_df['true_label'].to_numpy()
    y_prob = pred_df['predicted_prob'].to_numpy()
    nb11[defn] = {
        'metrics': evaluate(y_true, y_prob),
        'cis': bootstrap_ci(y_true, y_prob, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED),
    }
    m = nb11[defn]['metrics']
    ci = nb11[defn]['cis']
    print(f'[{defn}] nb11  AUROC={m["auroc"]:.4f} '
          f'[{ci["auroc"][0]:.4f},{ci["auroc"][1]:.4f}]  '
          f'AUPRC={m["auprc"]:.4f}  '
          f'sens@90spec={m["sensitivity_at_90_spec"]:.4f}')


[primary] nb11  AUROC=0.4688 [0.2031,0.7500]  AUPRC=0.6422  sens@90spec=0.0833
[pragmatic] nb11  AUROC=0.6880 [0.4473,0.8950]  AUPRC=0.7024  sens@90spec=0.1111


## 9 — Results summary

In [11]:
def _fmt(val, fmt='.4f'):
    if val is None:
        return 'N/A'
    v = float(val)
    return 'N/A' if (v != v) else format(v, fmt)

def _fmt_ci(ci_dict, key):
    if ci_dict is None:
        return 'N/A'
    lo, hi = ci_dict.get(key, (float('nan'), float('nan')))
    if lo != lo or hi != hi:
        return 'N/A'
    return f'[{lo:.4f}, {hi:.4f}]'

rows = []
for defn in DEFINITIONS:
    # nb11 baseline
    if nb11[defn] is not None:
        m, ci = nb11[defn]['metrics'], nb11[defn]['cis']
        rows.append({'Definition': defn, 'Design': 'nb11-baseline',
                     'AUROC': _fmt(m['auroc']), '95% CI': _fmt_ci(ci, 'auroc'),
                     'AUPRC': _fmt(m['auprc']),
                     'Sens@90Spec': _fmt(m['sensitivity_at_90_spec']),
                     'Perm p': 'N/A'})
    # D1 sex covariate
    if sex_cov_metrics[defn] is not None:
        m, ci = sex_cov_metrics[defn], sex_cov_cis[defn]
        rows.append({'Definition': defn, 'Design': 'D1-sex-cov',
                     'AUROC': _fmt(m['auroc']), '95% CI': _fmt_ci(ci, 'auroc'),
                     'AUPRC': _fmt(m['auprc']),
                     'Sens@90Spec': _fmt(m['sensitivity_at_90_spec']),
                     'Perm p': _fmt(sex_cov_perms[defn]['p_value'], '.3f')})
    else:
        rows.append({'Definition': defn, 'Design': 'D1-sex-cov (SKIPPED)',
                     'AUROC': 'SKIPPED', '95% CI': '', 'AUPRC': 'SKIPPED',
                     'Sens@90Spec': 'SKIPPED', 'Perm p': 'SKIPPED'})
    # D2 no-Y
    m, ci = noy_metrics[defn], noy_cis[defn]
    rows.append({'Definition': defn, 'Design': 'D2-no-Y-no-XCT',
                 'AUROC': _fmt(m['auroc']), '95% CI': _fmt_ci(ci, 'auroc'),
                 'AUPRC': _fmt(m['auprc']),
                 'Sens@90Spec': _fmt(m['sensitivity_at_90_spec']),
                 'Perm p': _fmt(noy_perms[defn]['p_value'], '.3f')})
    # D3 curated
    m, ci = curated_metrics[defn], curated_cis[defn]
    rows.append({'Definition': defn, 'Design': 'D3-curated-ALT',
                 'AUROC': _fmt(m['auroc']), '95% CI': _fmt_ci(ci, 'auroc'),
                 'AUPRC': _fmt(m['auprc']),
                 'Sens@90Spec': _fmt(m['sensitivity_at_90_spec']),
                 'Perm p': _fmt(curated_perms[defn]['p_value'], '.3f')})

results_df = pd.DataFrame(rows, columns=['Definition','Design','AUROC','95% CI',
                                          'AUPRC','Sens@90Spec','Perm p'])
print(results_df.to_string(index=False))


Definition               Design   AUROC           95% CI   AUPRC Sens@90Spec  Perm p
   primary        nb11-baseline  0.4688 [0.2031, 0.7500]  0.6422      0.0833     N/A
   primary           D1-sex-cov  0.5000 [0.1904, 0.8001]  0.5818      0.0000   0.530
   primary       D2-no-Y-no-XCT  0.3958 [0.1098, 0.7144]  0.5386      0.0000   0.660
   primary       D3-curated-ALT  0.9479 [0.8125, 1.0000]  0.9755      0.9167   0.000
 pragmatic        nb11-baseline  0.6880 [0.4473, 0.8950]  0.7024      0.1111     N/A
 pragmatic D1-sex-cov (SKIPPED) SKIPPED                  SKIPPED     SKIPPED SKIPPED
 pragmatic       D2-no-Y-no-XCT  0.7051 [0.4772, 0.8955]  0.7443      0.2222   0.060
 pragmatic       D3-curated-ALT  0.8291 [0.6681, 0.9625]  0.9166      0.7778   0.000


## 10 — Sex distribution per arm

In [12]:
for defn in DEFINITIONS:
    ids = filtered[defn]['sample_id'].tolist()
    sex_labels = [sex_lookup.get(sid, 'Unknown') for sid in ids]
    arm_labels = filtered[defn]['alt_call_direction'].tolist()
    tmp = pd.DataFrame({'Sex': sex_labels, 'alt_call_direction': arm_labels})
    print(f'\n[{defn}] N={len(ids)}')
    print(pd.crosstab(tmp['Sex'], tmp['alt_call_direction']))



[primary] N=20
alt_call_direction  negative  positive
Sex                                   
Female                     4         7
Male                       4         5

[pragmatic] N=31
alt_call_direction  negative  positive
Sex                                   
Female                     5         8
Male                       5         9
Unknown                    3         1


## 11 — Curated ALT gene coefficients

Coefficients from a full-data refit at the median LOOCV-selected C. Positive coefficient = higher expression associates with ALT+.

In [13]:
for defn in DEFINITIONS:
    model = curated_models[defn]
    coef = model['clf'].coef_.ravel()
    fixed_C = float(np.median(curated_loocv[defn].selected_Cs))
    coef_df = pd.DataFrame({'gene_symbol': curated_syms, 'coefficient': coef})
    coef_df['abs_coef'] = coef_df['coefficient'].abs()
    coef_df = coef_df.sort_values('abs_coef', ascending=False).reset_index(drop=True)
    coef_df.insert(0, 'rank', range(1, len(coef_df) + 1))
    print(f'\n[{defn}] Curated ALT gene coefficients  '
          f'(C={fixed_C:.4f}, {len(curated_syms)} genes)')
    print(coef_df[['rank','gene_symbol','coefficient']].to_string(index=False))



[primary] Curated ALT gene coefficients  (C=0.3162, 19 genes)
 rank gene_symbol  coefficient
    1        TERT    -0.689151
    2       FANCM     0.683066
    3       BRCA2     0.196771
    4         WRN    -0.152990
    5         NBN     0.105463
    6        TP53    -0.023244
    7        ATRX     0.000000
    8       RAD52     0.000000
    9        GEN1     0.000000
   10        SLX4     0.000000
   11       BRCA1     0.000000
   12      FANCD2     0.000000
   13      WRAP53     0.000000
   14    RAD51AP1     0.000000
   15        DAXX     0.000000
   16       RAD50     0.000000
   17         BLM     0.000000
   18    SMARCAL1     0.000000
   19       MUS81     0.000000

[pragmatic] Curated ALT gene coefficients  (C=0.3162, 19 genes)
 rank gene_symbol  coefficient
    1       FANCM     0.767177
    2        TERT    -0.754628
    3        TP53    -0.336636
    4         NBN     0.237621
    5         WRN    -0.219992
    6        ATRX    -0.124507
    7       BRCA2     0.116268
    

## 12 — Write outputs and report

In [14]:
# ── Prediction CSVs ─────────────────────────────────────────────────────────
output_paths = {}

def _write_pred_csv(res, label, defn):
    if res is None:
        return
    df = pd.DataFrame({'sample_id': res.sample_ids,
                       'true_label': res.y_true,
                       'predicted_prob': res.y_prob})
    p = PROC / f'aim2_sex_{label}_{defn}_predictions.csv'
    df.to_csv(p, index=False)
    output_paths[p.name] = p

for defn in DEFINITIONS:
    _write_pred_csv(sex_cov_loocv[defn], 'covariate', defn)
    _write_pred_csv(noy_loocv[defn], 'no_y', defn)
    _write_pred_csv(curated_loocv[defn], 'curated', defn)

for name in sorted(output_paths):
    print(f'  wrote {name}  ({output_paths[name].stat().st_size:,} bytes)')

# ── Markdown report ──────────────────────────────────────────────────────────
L = []
L.append('# Notebook 12 — Aim 2 ALT Signature: Sex-Adjusted Sensitivity Analysis\n\n')
L.append(f'Run timestamp: `{RUN_TIMESTAMP}`\n\n')
L.append('Sensitivity analysis for sex confounding in the notebook 11 primary top-20 '
         '(USP9Y, DDX3Y). Three designs x two definitions = 6 analyses.\n\n')

L.append('## Configuration\n\n')
L.append(f'- Gene variance filter: top `{TOP_K_VARIABLE_GENES}` genes (D1, D2); '
         f'bypassed for D3.\n')
L.append(f'- Model: `LogisticRegression(penalty="elasticnet", solver="saga", '
         f'l1_ratio={L1_RATIO}, class_weight="balanced", '
         f'random_state={RANDOM_SEED})`.\n')
L.append(f'- C grid: `10**linspace(-2,2,9)`.  '
         f'Inner CV folds: {INNER_CV_FOLDS}.\n')
L.append(f'- Bootstrap: {N_BOOTSTRAP} resamples.  '
         f'Permutations: {N_PERMUTATIONS} (fixed median C).\n\n')

L.append('## Results\n\n')
cols_md = ['Definition','Design','AUROC','95% CI','AUPRC','Sens@90Spec','Perm p']
L.append('| ' + ' | '.join(cols_md) + ' |\n')
L.append('|' + '|'.join(['---'] * len(cols_md)) + '|\n')
for row in rows:
    L.append('| ' + ' | '.join(str(row.get(c,'')) for c in cols_md) + ' |\n')
L.append('\n')

L.append('## Sex distribution per arm\n\n')
for defn in DEFINITIONS:
    ids = filtered[defn]['sample_id'].tolist()
    sex_labels = [sex_lookup.get(sid, 'Unknown') for sid in ids]
    arm_labels = filtered[defn]['alt_call_direction'].tolist()
    tmp = pd.DataFrame({'Sex': sex_labels, 'alt_call_direction': arm_labels})
    ct = pd.crosstab(tmp['Sex'], tmp['alt_call_direction'])
    L.append(f'### {defn.capitalize()} (N={len(ids)})\n\n')
    L.append('```\n' + ct.to_string() + '\n```\n\n')

L.append('## D1 — Sex covariate notes\n\n')
for defn in DEFINITIONS:
    if not sex_status[defn]['ok']:
        unk = sex_status[defn]['unknown']
        L.append(f'**{defn.capitalize()}**: SKIPPED — {len(unk)} sample(s) with '
                 f'Unknown/missing sex: {unk}\n\n')
    else:
        L.append(f'**{defn.capitalize()}**: Analysis completed. '
                 f'All {len(ids_all[defn])} samples have Female or Male annotation.\n\n')

L.append('## D2 — Excluded genes\n\n')
L.append(f'Total excluded: {len(exclude_names)} columns '
         f'({n_ychr_found} Y-chr + {n_xct_found} X CT antigens)\n\n')
L.append('```\n')
L.append('\n'.join(sorted(strip_gene_symbol(c) for c in exclude_names)))
L.append('\n```\n\n')

L.append('## D3 — Curated ALT gene availability\n\n')
L.append(f'Found: {len(curated_syms)}/{len(curated_syms_defined)} genes\n\n')
L.append(f'Not found in matrix: {not_found_curated}\n\n')

L.append('## D3 — Curated ALT gene coefficients\n\n')
for defn in DEFINITIONS:
    model = curated_models[defn]
    coef = model['clf'].coef_.ravel()
    fixed_C = float(np.median(curated_loocv[defn].selected_Cs))
    cdf = pd.DataFrame({'gene_symbol': curated_syms, 'coefficient': coef})
    cdf['abs_coef'] = cdf['coefficient'].abs()
    cdf = cdf.sort_values('abs_coef', ascending=False).reset_index(drop=True)
    L.append(f'### {defn.capitalize()} (C={fixed_C:.4f})\n\n')
    L.append('| Rank | Gene | Coefficient |\n|---|---|---|\n')
    for i, r in cdf.iterrows():
        L.append(f"| {i+1} | {r['gene_symbol']} | {r['coefficient']:+.6f} |\n")
    L.append('\n')

report_path = PROC / 'notebook_12_report.md'
report_path.write_text(''.join(L))
output_paths[report_path.name] = report_path
print(f'\nReport: {report_path.name}  ({report_path.stat().st_size:,} bytes)')


  wrote aim2_sex_covariate_primary_predictions.csv  (688 bytes)
  wrote aim2_sex_curated_pragmatic_predictions.csv  (1,035 bytes)
  wrote aim2_sex_curated_primary_predictions.csv  (679 bytes)
  wrote aim2_sex_no_y_pragmatic_predictions.csv  (1,039 bytes)
  wrote aim2_sex_no_y_primary_predictions.csv  (683 bytes)

Report: notebook_12_report.md  (3,675 bytes)


## 13 — SHA-256 idempotence check

In [15]:
SHA_PATH = PROC / 'notebook_12_sha256.json'
HASHED = sorted(output_paths.keys())
current = {name: sha256_file(output_paths[name]) for name in HASHED}
for name in HASHED:
    print(f'  {name}: {current[name]}')

if SHA_PATH.exists():
    previous = json.loads(SHA_PATH.read_text())
    mismatch = {
        k: (previous.get(k), current[k])
        for k in HASHED if previous.get(k) != current[k]
    }
    if mismatch:
        print('\nIDEMPOTENCE FAILED — outputs differ from recorded sha256:')
        for k, (was, now) in mismatch.items():
            print(f'  {k}: was {was} now {now}')
        raise AssertionError(f'Idempotence FAILED for: {sorted(mismatch)}')
    print('\nIdempotence check PASSED: all output files byte-identical to recorded sha256.')
else:
    print('\nFirst run: recording sha256 baseline (re-run to verify byte-identity).')

SHA_PATH.write_text(json.dumps(current, indent=2, sort_keys=True) + '\n')
print(f'sha256 manifest: {SHA_PATH.relative_to(ROOT)}')


  aim2_sex_covariate_primary_predictions.csv: 0b7e862f46e98441378e14f47f2ee1d4ee1774660f6d1ee0ca9f3a1f8ab5ce9a
  aim2_sex_curated_pragmatic_predictions.csv: b1fcdc215ad0817484897d7fc1a7e52bc73c7aac527622408ca48837ea8bf30d
  aim2_sex_curated_primary_predictions.csv: 62c18e5fe467891f131345ba5884de9e4a1f64f7ffd709a0763569c7f2173e76
  aim2_sex_no_y_pragmatic_predictions.csv: 62e13d69b15dfd28d674c701388978602a8efb4c4268ffa39975c5e13ee94a5e
  aim2_sex_no_y_primary_predictions.csv: 2d04940b2ba46e62816f1bc871df14acef51ca762e70b8fff7deb2ea25d0ac4e
  notebook_12_report.md: 24efc66362dfd6461824b5575308c0f1b08e4ecb4232d28952dbe49b0379a494

First run: recording sha256 baseline (re-run to verify byte-identity).
sha256 manifest: data/processed/notebook_12_sha256.json
